# BMRS – Indicated Imbalance & Wind/Solar

Both tasks, driven from the `energyviz` package rather than re-declared here.
Every function this notebook calls lives in a module that is unit-tested; the
notebook is only the narrative.

| Layer | What it does |
|---|---|
| `energyviz.sources` | talk to an API, return a frame shaped like `energyviz.schema` |
| `energyviz.domain`  | pure transforms on those frames |
| `energyviz.viz`     | frame → figure, and the one place that writes files |
| `energyviz.store`   | Parquet round-trip |


## Setup


In [ ]:
import sys, logging
sys.path.insert(0, '..')          # so the package is importable from notebooks/

logging.basicConfig(level=logging.INFO, format='%(message)s')

from energyviz import schema, sources
from energyviz.config import Settings
from energyviz.domain import imbalance, windsolar
from energyviz.store import parquet
from energyviz.viz import export, figures, theme

settings = Settings()            # base URL, Europe/Berlin, retry policy
client = sources.build('elexon', settings)

imbalance_date = '2025-12-07'
windsolar_date = '2025-11-11'


## Task 1 — indicated imbalance

One local (CE(S)T) day is settlement periods 47–48 of the previous UTC date
followed by 1–46 of the selected one. `fetch_imbalance_local_day` issues both
requests and returns a single normalised frame.


In [ ]:
raw = client.fetch_imbalance_local_day(imbalance_date)
raw.head()


The evolution feed republishes a period every time the forecast moves, so the
day arrives as thousands of rows across four indicated quantities.


In [ ]:
raw.groupby('series').size()


`latest_per_period` keeps the most recently published row per period, and
`add_sign` labels each one for colouring.


In [ ]:
snapshot = imbalance.add_sign(imbalance.latest_per_period(raw))
snapshot[['settlement_period', 'value', 'value_sign', 'publish_utc']].head()


In [ ]:
fig = figures.imbalance_snapshot(snapshot, settings.timezone)
fig.show()


### How the forecast moved

`diff_snapshots` lines two snapshots up and measures the revision. Re-run the
fetch after BMRS has republished to compare two real snapshots; the cell below
shifts the values so the plot has something to draw.


In [ ]:
previous = snapshot.copy()
previous['value'] = previous['value'] * 0.85

merged, same_date = imbalance.diff_snapshots(previous, snapshot)
title = figures.diff_title(previous, snapshot, same_date, settings.timezone)

figures.imbalance_diff(merged, title=title).show()


## Task 2 — wind & solar, forecast vs actual


In [ ]:
forecast = client.fetch_wind_solar_local_day(windsolar_date, schema.generation_forecast)
actual = client.fetch_wind_solar_local_day(windsolar_date, schema.generation_actual)

aligned = windsolar.align(forecast, actual)
aligned.head()


Wind arrives split into onshore and offshore; `align` sums them per period
before joining, so `forecast_mw` and `actual_mw` are directly comparable.


In [ ]:
wind = windsolar.split(aligned, schema.wind)
solar = windsolar.split(aligned, schema.solar)

figures.forecast_vs_actual(wind, 'Wind').show()


In [ ]:
figures.forecast_vs_actual(solar, 'Solar').show()


Plotting against local time instead of settlement period is a keyword, not a
different function.


In [ ]:
figures.forecast_vs_actual(wind, 'Wind', x_axis='start_local').show()


### Forecast error


In [ ]:
for df, label in ((wind, 'Wind'), (solar, 'Solar')):
    summary = windsolar.error_summary(df, label)
    print('\n'.join(summary.lines()))


## Saving

`viz.export` is the only module in the package that writes plot files, and
`store.parquet` the only one that writes data. Both take the directory; nothing
upstream of them knows about the filesystem.


In [ ]:
export.save(figures.forecast_vs_actual(wind, 'Wind'), 'out', 'wind_forecast_vs_actual',
            height=export.height_for_table(len(wind)))


In [ ]:
import pandas as pd

parquet.write(pd.concat([forecast, actual], ignore_index=True), 'out/data')
parquet.read('out/data', dataset=schema.generation_actual).head()


## Restyling

The palette is a value, so a second theme needs no edit to any plot function.


In [ ]:
midnight = theme.Theme(
    paper_bg='#12161c', plot_bg='#12161c',
    grid='#232a33', axis='#4a5866', tick='#c8d2dc',
    green='#4ade80', red='#f87171',
    font_family='Inter, system-ui, sans-serif',
)

figures.imbalance_snapshot(snapshot, settings.timezone, theme=midnight).show()
